In [ ]:
import torch
import torch.nn as nn
import numpy as np

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]= "0"

In [ ]:
class PromoptTuning(nn.Module):
    def __init__(self, num_prompt_tokens, d_model):
        super().__init__()
        self.soft_prompt = nn.Parameter(torch.randn(num_prompt_tokens, d_model))

    def forward(self, input_embeddings):
        """
        input_embedings : (batch_size, seq_len, d_model)
        return : (batch_size, num_prompt_tokens + seq_len, d_model)
        """

        batch_size = input_embeddings.shape[0]
        prompt = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        print(prompt.shape)
        return torch.cat([prompt, input_embeddings], dim=1)


In [ ]:
d_model = 64
prompt_tuning = PromoptTuning(num_prompt_tokens=5, d_model=d_model)

fake_input = torch.randn(2, 10, d_model)
output = prompt_tuning(fake_input)

output.shape

torch.Size([2, 5, 64])


torch.Size([2, 15, 64])

In [ ]:
# x = torch.randn(3, 2)
# nn.Linear()(x)  : x라는 텐서에 가중치를 준다

In [ ]:
num_classes = 2
embedding = nn.Embedding(50, d_model)
prompt_tuning = PromoptTuning(num_prompt_tokens=5, d_model=d_model)
classifier = nn.Linear(d_model, num_classes)

for p in embedding.parameters():
    p.requires_grad = False
for p in classifier.parameters():
    p.requires_grad = False

train_inputs = torch.randint(0, 50, (20, 8))
train_labels= torch.randint(0, 2, (20,))

optimizer = torch.optim.Adam([prompt_tuning.soft_prompt], lr=0.01)
criterion = nn.CrossEntropyLoss()



In [ ]:
initial_prompt = prompt_tuning.soft_prompt.data.clone()

for epoch in range(10):
    emb = embedding(train_inputs)
    emb_with_prompt = prompt_tuning(emb)
    pooled = emb_with_prompt.mean(dim=1)
    logits = classifier(pooled)
    loss = criterion(logits, train_labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 3 == 0:
        print(f"Epoch : {epoch}")



torch.Size([20, 5, 64])
Epoch : 0
torch.Size([20, 5, 64])
torch.Size([20, 5, 64])
torch.Size([20, 5, 64])
Epoch : 3
torch.Size([20, 5, 64])
torch.Size([20, 5, 64])
torch.Size([20, 5, 64])
Epoch : 6
torch.Size([20, 5, 64])
torch.Size([20, 5, 64])
torch.Size([20, 5, 64])
Epoch : 9


In [ ]:
(prompt_tuning.soft_prompt.data - initial_prompt).norm()

tensor(1.7141)

In [ ]:
# initial_prompt

In [ ]:
# prompt_tuning.soft_prompt.data

In [ ]:
d_model = 768
for num_tokens in [5, 10, 20, 50, 100]:
    prompt_params = num_tokens * d_model
    print(f"prompt token : {num_tokens}, prompt params : {prompt_params}")

prompt token : 5, prompt params : 3840
prompt token : 10, prompt params : 7680
prompt token : 20, prompt params : 15360
prompt token : 50, prompt params : 38400
prompt token : 100, prompt params : 76800


In [ ]:
# soft prompting 구현
# 1. d_model = 16
# 2. num_tokens = 4
d_model = 16
prompt_tuning = PromoptTuning(num_prompt_tokens=4, d_model=d_model)

fake_input = torch.randn(2, 10, d_model)
output = prompt_tuning(fake_input)



torch.Size([2, 4, 16])


In [ ]:
num_classes = 2
embedding = nn.Embedding(50, d_model)
prompt_tuning = PromoptTuning(num_prompt_tokens=4, d_model=d_model)
classifier = nn.Linear(d_model, num_classes)

for p in embedding.parameters():
    p.requires_grad = False
for p in classifier.parameters():
    p.requires_grad = False

train_inputs = torch.randint(0, 50, (20, 8))
train_labels= torch.randint(0, 2, (20,))

optimizer = torch.optim.Adam([prompt_tuning.soft_prompt], lr=0.01)
criterion = nn.CrossEntropyLoss()



In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PromptTuningConfig, PromptTuningInit, get_peft_model, TaskType

/home/oncreative/.local/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [ ]:
MODEL_ID = "klue/bert-base"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

In [ ]:
pt_config = PromptTuningConfig(
    task_type = TaskType.SEQ_CLS,
    num_virtual_tokens = 5,
    prompt_tuning_init = PromptTuningInit.RANDOM
)

In [ ]:
pt_model = get_peft_model(base_model, pt_config)

In [ ]:
pt_model.print_trainable_parameters()

trainable params: 3,840 || all params: 110,622,722 || trainable%: 0.0035


In [ ]:
pt_config.num_virtual_tokens

5

In [ ]:
pt_config.prompt_tuning_init

<PromptTuningInit.RANDOM: 'RANDOM'>

In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments, Trainer

nsmc = load_dataset('nsmc')
train_ds = nsmc['train'].shuffle().select(range(2000))
test_ds = nsmc['test'].shuffle().select(range(500))


In [ ]:
def tokenize_fn(examples):
    return tokenizer(examples['document'], padding='max_length', truncation=True, max_length=128)

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns = ['id', 'document'])
test_tok = test_ds.map(tokenize_fn, batched=True, remove_columns = ['id', 'document'])
train_tok.set_format('torch')
test_tok.set_format('torch')

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
train_tok.column_names

['label', 'input_ids', 'token_type_ids', 'attention_mask']

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy" : accuracy_score(labels, preds)}

In [ ]:
training_args = TrainingArguments(
    output_dir = './prompt_tuning_output',
    num_train_epochs = 3,
    learning_rate = 1e-3,
    fp16=True
)

trainer = Trainer(
    model = pt_model,
    args = training_args,
    train_dataset = train_tok,
    eval_dataset = test_tok,
    compute_metrics = compute_metrics
)

trainer.train()

Step,Training Loss
500,0.680780


TrainOutput(global_step=750, training_loss=0.6797121988932292, metrics={'train_runtime': 35.5617, 'train_samples_per_second': 168.721, 'train_steps_per_second': 21.09, 'total_flos': 394666583040000.0, 'train_loss': 0.6797121988932292, 'epoch': 3.0})

In [ ]:
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

result = trainer.evaluate()

In [ ]:
result

{'eval_loss': 0.6827250719070435,
 'eval_accuracy': 0.548,
 'eval_runtime': 0.8139,
 'eval_samples_per_second': 614.314,
 'eval_steps_per_second': 77.404,
 'epoch': 3.0}

In [ ]:
# prefix tuning
# prompt tuning :
# 나는 학교에 갑니다 -> [p1, p2, p3, p4, p5, t1, ,,,, tn] -> model(layer1, ...) -> output

# prefix tuning
# [t1, ... tn] -> [p1, p2, t1, ... tn] -> layr1 -> [p1`, p2`, t1` , ... tn`] ->

# layer1 : k, v [pk1, k],[pv1, v] ->
# layer2 : k, v [pk2, k],[pv2, v]

In [ ]:
d_model = 64
num_layers = 6
num_tokens = 10

prompt_tuning_prams = nn.Parameter(torch.randn(num_tokens, d_model))

prefix_params = nn.ParameterList([
    nn.Parameter(torch.randn(num_tokens, d_model)) for _ in range(num_layers * 2)])

In [ ]:
d_k = 8   #128 512
seq_len = 4   # 1024
prefix_len = 2

Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

P_K = nn.Parameter(torch.randn(prefix_len, d_k))
P_V = nn.Parameter(torch.randn(prefix_len, d_k))

K_with_prefix = torch.cat([P_K,K],dim=0)
V_with_prefix = torch.cat([P_V,V],dim=0)

print(f"원래 : {K.shape} -> {K_with_prefix.shape}")
print(f"Q : {Q.shape}")

원래 : torch.Size([4, 8]) -> torch.Size([6, 8])
Q : torch.Size([4, 8])


In [ ]:
scores = Q @ K_with_prefix.T / (d_k ** 0.5)
print(scores.shape)
weight = torch.softmax(scores, dim=1)
print(weight.shape)
output = weight @ V_with_prefix
print(output.shape)

torch.Size([4, 6])
torch.Size([4, 6])
torch.Size([4, 8])


In [ ]:
class ReprameterPrefix(nn.Module):
    def __init__(self, prefix_len, d_model, bottleneck=32):
        super().__init__()
        self.embedding = nn.Embedding(prefix_len, bottleneck)
        self.mlp = nn.Sequential(
                nn.Linear(bottleneck, d_model),
                nn.Tanh(),
                nn.Linear(d_model, d_model),
        )
        self.prefix_ids = torch.arange(prefix_len)

    def forward(self):
        small = self.embedding(self.prefix_ids)
        prefix = self.mlp(small)
        return prefix

In [ ]:
prefix_gen = ReprameterPrefix(prefix_len = 10, d_model=64, bottleneck=32)
prefix_vectors = prefix_gen()

In [ ]:
# prefix_vectors

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PrefixTuningConfig, get_peft_model, TaskType

In [ ]:
# "klue/bert-base"
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
prefix_config = PrefixTuningConfig(
    task_type = TaskType.SEQ_CLS,
    num_virtual_tokens = 10,
    prefix_projection=True,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

In [ ]:
prefix_model = get_peft_model(base_model, prefix_config)

In [ ]:
prefix_model

PeftModelForSequenceClassification(
  (base_model): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(32000, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_featur

In [ ]:
training_args = TrainingArguments(
    output_dir = './prefix_tuning_output',
    num_train_epochs = 3,
    learning_rate = 1e-3,
    fp16=True
)

trainer = Trainer(
    model = prefix_model,
    args = training_args,
    train_dataset = train_tok,
    eval_dataset = test_tok,
    compute_metrics = compute_metrics
)

trainer.train()

AttributeError: type object 'DynamicCache' has no attribute 'from_legacy_cache'

In [ ]:
!pip install --upgrade transformers peft

  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
Using cached peft-0.18.1-py3-none-any.whl (556 kB)
  Attempting uninstall: peft
    Found existing installation: peft 0.15.2
    Uninstalling peft-0.15.2:
      Successfully uninstalled peft-0.15.2
